# Функциональное программирование

## Домашнее задание 5

**1.** Реализуйте функцию, которая запрашивает у пользователя количество чисел, затем сами числа (с плавающей запятой), а затем предлагает «меню»: требуется ли сложить или перемножить данные числа. Наконец, выводится соответствующий результат.

In [6]:
import Control.Monad (replicateM)

readDoubles :: Int -> IO [Double]
readDoubles n = replicateM n (fmap read getLine)

-- Меню: '+' — сумма, '*' — произведение
menu :: [Double] -> IO Double
menu xs = do
  putStrLn "Выберите операцию: '+' для суммы, '*' для произведения"
  op <- getLine
  case op of
    "+" -> return (sum xs)
    "*" -> return (product xs)
    _   -> do
      putStrLn "Неизвестная операция, попробуйте ещё раз."
      menu xs

task1 :: IO ()
task1 = do
  putStrLn "Сколько чисел?"
  n <- fmap read getLine
  putStrLn "Введите числа (по одному в строке):"
  xs <- readDoubles n
  res <- menu xs
  putStrLn ("Результат: " ++ show res)

-- Проверить в ноутбуке не получится, надо сохранить как отдельный файл и запустить уже так

**2.** Реализуйте программу, которая получает из командной строки пути к входному и выходному файлу, а также бинарное значение. Входной файл открывается для чтения, и все его строки (lines) лексикографически упорядочиваются по невозрастанию (или по неубыванию — в зависимости от бинарного значения). Результат записывается в выходной файл. Обработка ошибок (таких как отсутствие входного файла) должна быть разумной.

In [7]:
import System.Environment (getArgs)
import System.IO (hPutStrLn, stderr)
import Control.Exception (catch, IOException)
import Data.List (sort, sortBy)
import Data.Ord (comparing, Down(..))

-- True — по неубыванию, False — по невозрастанию
sortFile :: FilePath -> FilePath -> Bool -> IO ()
sortFile inp out asc = do
  contents <- readFile inp
  let ls = lines contents
      sorted = if asc then sort ls else sortBy (comparing Down) ls
  writeFile out (unlines sorted)

task2 :: IO ()
task2 = do
  args <- getArgs
  case args of
    [inp, out, flag] -> do
      let asc = flag `elem` ["1", "true", "True", "asc"]
      sortFile inp out asc
        `catch` \e -> hPutStrLn stderr ("Ошибка ввода-вывода: " ++ show (e :: IOException))
    _ -> hPutStrLn stderr "Использование: task2 <входной файл> <выходной файл> <0|1>" 

-- Проверка на временных файлах
writeFile "/tmp/in.txt" "banana\napple\ncherry\n"
sortFile "/tmp/in.txt" "/tmp/out_asc.txt" True
sortFile "/tmp/in.txt" "/tmp/out_desc.txt" False
readFile "/tmp/out_asc.txt"
readFile "/tmp/out_desc.txt" 

"apple\nbanana\ncherry\n"

"cherry\nbanana\napple\n"

**3.** Реализуйте функцию, вычисляющую «случайную» перестановку данного списка. Каждый новый вызов функции должен давать результат, почти наверное отличный (в идеале независимый) от результата предыдущего вызова. В идеале все возможные перестановки равновероятны. Поведение на бесконечных списках неопределено.

Каждому элементу сопоставим случайное число и отсортируем по нему. При непрерывном распределении ключей все $n!$ перестановок равновероятны.

In [8]:
import System.Random (randomRs, newStdGen)
import Data.List (sortBy)
import Data.Ord (comparing)

shuffle :: [a] -> IO [a]
shuffle xs = do
  g <- newStdGen
  let keys = take (length xs) (randomRs (0 :: Double, 1) g)
  return (map snd (sortBy (comparing fst) (zip keys xs)))

-- Проверка
shuffle [1..10 :: Int]
shuffle [1..10 :: Int]
shuffle "abcdefgh" 

[5,2,7,4,9,6,10,8,1,3]

[3,7,6,9,2,4,5,1,10,8]

"gbdhcefa"

**4.** Рассмотрим шар $S=\{(x,y,z)\in\mathbb{R}^3 \mid x^2+y^2+z^2\leq 1\}$, вписанный в куб $C=\{(x,y,z)\in\mathbb{R}^3 \mid \max(|x|,|y|,|z|)\leq 1\}$. Зная соотношение объемов куба и шара, оцените число $\pi$ через долю «случайных» точек куба, оказывающихся внутри шара. Количество точек должно быть аргументом вашей функции.

Объём куба равен $8$, объём шара — $\tfrac{4}{3}\pi$, их отношение равно $\pi/6$. Если доля точек, попавших в шар, есть $p$, то $\pi\approx 6p$.

In [9]:
import System.Random (randomRs, newStdGen)

estimatePi :: Int -> IO Double
estimatePi n = do
  g <- newStdGen
  let coords = take (3 * n) (randomRs (-1 :: Double, 1) g)
      triples [] = []
      triples (x:y:z:rest) = (x, y, z) : triples rest
      triples _ = []
      inside = length [() | (x, y, z) <- triples coords, x*x + y*y + z*z <= 1]
  return (6 * fromIntegral inside / fromIntegral n)

-- Проверка
estimatePi 1000
estimatePi 100000
estimatePi 1000000

3.276

3.135

3.137592

**5.** Реализуйте функцию `average :: (Foldable f, Fractional a) => f a -> Maybe a`, вычисляющую среднее значение элементов коллекции, используя не более одной свёртки контейнера.

In [10]:
average :: (Foldable f, Fractional a) => f a -> Maybe a
average xs = case foldr step (0, 0 :: Int) xs of
  (_, 0) -> Nothing
  (s, n) -> Just (s / fromIntegral n)
 where
  step x (s, n) = (s + x, n + 1)

-- Проверка
average ([] :: [Double])
average [1, 2, 3, 4, 5 :: Double]
average (Just 7 :: Maybe Double)
average [1.5, 2.5, 3.5 :: Double]

Nothing

Just 3.0

Just 7.0

Just 2.5